# Classification

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

### Local imports

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

print("Added to path:", project_root)  # verify it looks right


from models.decision_tree import FCDecisionTreeClassifier, crossval_fc_trees
from models.random_forest import FCRandomForestClassifier

from src.data_loader import load_zFC_df

## Hyperparams

In [ ]:
# restAP: 1 - restPA: 2 - all: 3
task_case = 2
# full: 1 - all: 2 - slow3: 3 - slow4: 4 - slow5 - 5
slow_case = 5
both_runs = True

# tree params
seed = 42
max_depth = 10
ccp_alpha = 0.005

#region config globals
if task_case == 1:
    task_type = 'restAP'
elif task_case == 2:
    task_type = 'restPA'
elif task_case == 3:
    task_type = 'all'
else:
    raise ValueError("Task case not valid. 1, 2 and 3 are valid values.")
if slow_case == 1:
    band = 'full'
elif slow_case == 2:
    band = 'all'
elif slow_case in [3, 4, 5]:
    band = 'slow' + str(slow_case)
else:
    raise ValueError("Band case not valid. 1, 2, 3, 4 and 5 are valid values.")
runs = 2 if both_runs else 1
#endregion

In [ ]:
def test_decision_tree(b, t, i):
    DTclf = FCDecisionTreeClassifier(band_type=b, task_type=t, max_depth=max_depth, random_state=seed, ccp_alpha=ccp_alpha)

    data = load_zFC_df(band_type=DTclf.band_type, task_type=DTclf.task_type, runs=1)
    X = data.iloc[:, 0:-1]
    y = data.MDD

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test = pd.DataFrame(scaler.fit_transform(X_test), columns=X.columns)

    DTclf.fit(X_train, y_train)
    DTclf.evaluate(X_test, y_test)
    DTclf.save()

    plot_tree(DTclf, filled=True, ax=i)

    return DTclf.plot_decision_paths(X=X_test, return_png=True)

In [ ]:
FCDTs = {}

fig, axs = plt.subplots(1, 12, figsize=(120, 10))
i = 0

for t in ['restAP', 'restPA', 'all']:
    for b in ['slow3', 'slow4', 'slow5', 'full']:
        if t == 'all':
            key = b
        elif b == 'full':
            key = t[-2:] + '_' + b
        else:
            key = t[-2:] + '_s' + b[-1]
        print(f"--Running DT for band={b}, task={t}")
        FCDTs[key] = test_decision_tree(b, t, axs[i])
        i+=1
    print()

fig.tight_layout()
fig.show()

In [ ]:
df = crossval_fc_trees()

In [ ]:
print(df[df['band_type'] == 'slow3'].to_string())

In [ ]:
def test_random_forest(b, t, i):
    RFclf = FCRandomForestClassifier(band_type=b, task_type=t, max_depth=max_depth, random_state=seed, ccp_alpha=ccp_alpha)

    data = load_zFC_df(band_type=RFclf.band_type, task_type=RFclf.task_type, runs=1)
    X = data.iloc[:, 0:-1]
    y = data.MDD

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test = pd.DataFrame(scaler.fit_transform(X_test), columns=X.columns)

    RFclf.fit(X_train, y_train)
    RFclf.evaluate(X_test, y_test)
    RFclf.save()

In [ ]:
from models.random_forest import crossval_fc_forest

In [ ]:
df = crossval_fc_forest(
    param_grid={
        'n_estimators':     [100, 200, 300],
        'max_depth':        [1, 2, 3, 4],
        'min_samples_leaf': [1, 2, 3, 5],
    },
)

In [ ]:
print(df)

In [ ]:
print(df.to_string())

In [ ]:
import seaborn as sns
import numpy as np

def plot_param_distribution(
    df,
    param_name,
    score_col="f1",
    model_col="n_estimators",
    top_p=None,
    top_k=None,
    plot_points: bool = True,
):
    param_col = f"{param_name}"

    plot_df = df.copy()
    # plot_df[param_col] = plot_df[param_col].astype(str)
    # var = 0.01
    # plot_df[score_col] += np.random.normal(0, var, size=(len(plot_df),))

    if top_k is not None or top_p is not None:
        plot_df = plot_df.sort_values(score_col, ascending=False)

        if top_p is not None:
            n = max(1, int(len(plot_df) * top_p))
            plot_df = plot_df.head(n)

        if top_k is not None:
            plot_df = plot_df.head(top_k)

    plt.figure(figsize=(9, 5))

    sns.violinplot(
        data=plot_df,
        x=param_col,
        y=score_col,
        hue=model_col,
        # showfliers=False,
        fill=False,
        linewidth=0.5,
    )

    if plot_points:
        sns.swarmplot(
            data=plot_df,
            x=param_col,
            y=score_col,
            hue=model_col,
            dodge=True,
            # jitter=True,
            alpha=0.35,
            size=4,
            legend=False,
        )

    plt.xlabel(param_name)
    plt.ylabel(score_col)
    plt.title(f"Marginal effect of {param_name}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
df1 = df.loc[
    (df['task_type'] == 'restPA')
    #  & (df['band_type'] == 'slow5') 
    #  & (df['min_samples_leaf'] > 4) 
    #  & (df['n_estimators'] > 50)
    #  & (df['ccp_alpha'] > 0)
    #  & (df['max_depth'] > 2)
]
df1 = df1.astype({'max_depth': str})
t_k = 21600

for p, h in [
    ('max_depth', 'n_estimators'),
    # ('n_estimators', 'ccp_alpha'),
    # ('ccp_alpha', 'min_samples_leaf'),
    ('n_estimators', 'min_samples_leaf'),
    ('min_samples_leaf', 'max_depth')
]:

    plot_param_distribution(
        df=df1,
        param_name=p,
        score_col='f1',
        model_col=h,
        top_k=t_k,
        plot_points=True
    )

In [ ]:
df_copy = df.loc[
    (df['task_type'] == 'restPA')
    # & (df['band_type'] != 'full')
    # & (df['min_samples_leaf'] >= 5)
    # & (df['max_depth'] == 10)
    # & (df['n_estimators'] == 100)
    # & (df['accuracy'] > .5)
]
df_copy[['accuracy', 'precision', 'recall', 'f1']] += np.random.normal(0, 0.00, size=(len(df_copy), 4))
df_copy.loc[(df['max_depth'] == 'nan')] = 15

sns.pairplot(data=df_copy[['band_type', 'accuracy', 'precision', 'n_estimators', 'recall', 'f1']], hue='band_type', kind='reg')
sns.pairplot(data=df_copy[['band_type', 'accuracy', 'precision', 'min_samples_leaf', 'recall', 'f1']], hue='band_type', kind='reg')
sns.pairplot(data=df_copy[['band_type', 'accuracy', 'precision', 'max_depth', 'recall', 'f1']], hue='band_type', kind='reg')
# sns.pairplot(data=df_copy[['band_type', 'accuracy', 'precision', 'ccp_alpha', 'recall', 'f1']], hue='band_type', kind='reg')

In [ ]:
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from models.graph_neural_net import load_fc_graph_dataset

dataset = load_fc_graph_dataset(
    task_type="restPA",
    band_type="slow4",
    runs=2,
    atlas_type="Yan2023",
    network_means=False,
    decomp_method="memd",
    threshold=0.2,
)

labels = [graph.y.item() for graph in dataset]

train_val_graphs, test_graphs = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

train_graphs, val_graphs = train_test_split(
    train_val_graphs,
    test_size=0.25,  # 0.25 x 0.8 = 0.2
    random_state=42,
    stratify=[int(graph.y.item()) for graph in train_val_graphs],
)

train_loader = DataLoader(
    train_graphs,
    batch_size=7,
    shuffle=True,
)

val_loader = DataLoader(
    val_graphs,
    shuffle=False,
)

test_loader = DataLoader(
    test_graphs,
    shuffle=False,
)

In [ ]:
from models.graph_neural_net import FCGCN
import torch
from torchmetrics.classification import BinaryAccuracy, BinaryF1Score

n_epochs = 40
example_batch = next(iter(train_loader))

model = FCGCN(
    num_node_features=train_loader.dataset[0].num_node_features,
    hidden_channels=64,
    num_classes=2,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 1e-3,
    weight_decay=1e-4,
)

model.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=n_epochs,
)

model.eval()

# criterion = torch.nn.CrossEntropyLoss()

# train_acc_metric = BinaryAccuracy()
# train_f1_metric = BinaryF1Score()

# for epoch in range(n_epochs):
#     model.train()
    
#     train_acc_metric.reset()
#     train_f1_metric.reset()
    
#     train_loss = 0.0

#     for batch in iter(train_loader):

#         optimizer.zero_grad()

#         logits = model.forward(batch)
#         loss = criterion(logits, batch.y)

#         loss.backward()
#         optimizer.step()

#         train_loss += loss.item()

#         preds = logits.argmax(dim=1)

#         train_acc_metric.update(preds, batch.y)
#         train_f1_metric.update(preds, batch.y)
    
#     if (epoch+1) % 10 == 0:
#         train_loss /= len(train_loader)
#         train_acc = train_acc_metric.compute().item()
#         train_f1 = train_f1_metric.compute().item()

#         print(
#             f"Epoch {(epoch+1):03d} | "
#             f"Loss: {train_loss:.4f} | "
#             f"Acc: {train_acc:.3f} | "
#             f"f1: {train_f1:.3f}"
#         )

# for batch in iter(test_loader):
#     preds = model.predict(batch, return_probabilities=False)
#     acc = (preds == batch.y).float().mean().item()
#     print("Preds: ", preds)
#     print("Truth: ", batch.y)
#     print("Acc:", acc)